# English to French Machine Translation Comparison
**Yamir Singh Bamel · 220911280 · ICT4403 Assignment 3**

This notebook trains a basic Transformer, fine-tunes mT5, and prompts LLaMA and Gemma.
Select **Runtime → Change runtime type → GPU**, then run in order.
Full benchmark results are not prefilled. All outputs are computed by the cells below.

Before running, obtain access using your own Hugging Face account:
- https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
- https://huggingface.co/google/gemma-3-1b-it

A GPU session may take substantial time. Optional Drive storage can be enabled below to retain files.


In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile, io, base64
USE_DRIVE = False  # Set True to retain checkpoints in Drive (requires your authorization).
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/FGAI_Assignment3_220911280')
else:
    PROJECT = Path('/content/FGAI_Assignment3_220911280')
PROJECT.mkdir(parents=True, exist_ok=True)
SOURCE_ARCHIVE = '...'

# Preserve completed work and local edits on rerun.
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ARCHIVE))) as archive:
    for member in archive.infolist():
        target = PROJECT / member.filename
        if not target.exists():
            archive.extract(member, PROJECT)
os.chdir(PROJECT)
os.environ['HF_HOME'] = '/content/fgai_hf_cache'
os.environ['MPLCONFIGDIR'] = '/content/fgai_matplotlib'
print('Project:', PROJECT)


In [ ]:
# Dedicated text-only runtime: remove unused packages tied to a different Torch build.
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'], check=True)
requirements = Path('requirements.txt')
requirements.write_text(requirements.read_text().replace('sentencepiece==0.2.0', 'sentencepiece==0.2.1'))
installation = subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(installation.stdout)
installation.check_returncode()
# Keep subprocesses separate so each model releases GPU memory after its stage.
def run(*args):
    subprocess.run([sys.executable, *args], cwd=PROJECT, check=True)
run('-c', "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime before full training'; print(torch.cuda.get_device_name())")
run('-m', 'pytest', '-q')
with open(PROJECT / 'results' / 'environment.txt', 'w') as f:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=f, check=True)


## Private model login
Enter a read-only Hugging Face token in the hidden input. The token is held in the
session environment, not written to the project. Model access must already be approved.
If this cell fails, resolve access before spending time training.


In [ ]:
from getpass import getpass
os.environ['HF_TOKEN'] = getpass('Hugging Face read token: ')
run('-c', """
import json, os
from huggingface_hub import hf_hub_download
cfg = json.load(open('config.json'))
for name in ['llama', 'gemma']:
    hf_hub_download(cfg[name]['id'], 'tokenizer_config.json', token=os.environ['HF_TOKEN'])
    print(name, 'access verified')
""")


## Prepare the data
Fixed seed, normalized text, duplicate removal and disjoint sentence splits.
Inspect `results/dataset.json` after this cell. Related passages from the same book
may still occur in multiple splits.


In [ ]:
if not all((PROJECT / 'data' / f'{s}.json').exists() for s in ['train', 'validation', 'test']):
    run('-m', 'mt.prepare')
import json
print(json.loads((PROJECT / 'results/dataset.json').read_text()))


## Train and fine-tune
These are the longest stages. A completed stage is skipped when its training metadata
exists. If a stage was interrupted, its checkpoint directory remains; explicitly remove
only that incomplete directory before rerunning that stage. The program refuses to
overwrite checkpoints silently. No mid-epoch optimizer resume is implemented.


In [ ]:
if not (PROJECT / 'results/basic_training.json').exists():
    run('-m', 'mt.train', 'basic')
else:
    print('basic: using completed training stage')


In [ ]:
if not (PROJECT / 'results/mt5_training.json').exists():
    run('-m', 'mt.train', 'mt5')
else:
    print('mt5: using completed training stage')


## Evaluate all four systems
Every system receives the same test sources. Greedy decoding is used for the main
comparison. Each decoder-only stage also runs the temperature study. Warm-up uses a
validation sentence. Actual translations and qualitative probe outputs are saved.


In [ ]:
expected = ['basic_metrics.json', 'basic_predictions.json', 'basic_ethics.json']
if not all((PROJECT / 'results' / f).exists() for f in expected):
    run('-m', 'mt.evaluate', 'basic')
else:
    print('basic: using completed evaluation stage')


In [ ]:
expected = ['mt5_metrics.json', 'mt5_predictions.json', 'mt5_ethics.json']
if not all((PROJECT / 'results' / f).exists() for f in expected):
    run('-m', 'mt.evaluate', 'mt5')
else:
    print('mt5: using completed evaluation stage')


In [ ]:
expected = ['llama_metrics.json', 'llama_predictions.json', 'llama_ethics.json', 'llama_temperature_summary.json']
if not all((PROJECT / 'results' / f).exists() for f in expected):
    run('-m', 'mt.evaluate', 'llama')
else:
    print('llama: using completed evaluation stage')


In [ ]:
expected = ['gemma_metrics.json', 'gemma_predictions.json', 'gemma_ethics.json', 'gemma_temperature_summary.json']
if not all((PROJECT / 'results' / f).exists() for f in expected):
    run('-m', 'mt.evaluate', 'gemma')
else:
    print('gemma: using completed evaluation stage')


## Generate and inspect the report
The report refuses to invent missing results. It checks that all four test hashes
match, writes comparison tables/plots and updates README. Complete the human review
of actual translations and ethical examples before submission.


In [ ]:
run('-m', 'mt.report')
from IPython.display import display, Markdown, Image
import pandas as pd
display(pd.read_csv(PROJECT / 'results/comparison.csv'))
display(Image(filename=str(PROJECT / 'results/comparison.png')))
display(Image(filename=str(PROJECT / 'results/temperature.png')))
display(Markdown((PROJECT / 'results/analysis.md').read_text()))
display(pd.read_csv(PROJECT / 'results/samples.csv').head(15))
display(pd.read_csv(PROJECT / 'results/ethics_review.csv'))


## Download the repository files
The archive includes measured results and excludes raw training data, tokens, caches
and large checkpoints. Upload the extracted source and results to GitHub. Add your
repository URL to the supplied Word submission sheet, save it as PDF, and submit in LMS.
Do not submit until the ethical review and comparison have been checked.


In [ ]:
from google.colab import files
archive_path = Path('/content/FGAI_Assignment3_220911280_results.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in PROJECT.rglob('*'):
        relative = path.relative_to(PROJECT)
        if path.is_file() and not any(part.startswith('.') or part in ['data', 'checkpoints', '__pycache__'] for part in relative.parts):
            if path.suffix != '.zip':
                archive.write(path, relative)
files.download(str(archive_path))
